<a href="https://colab.research.google.com/github/Eddythemachine/coursera_neural_network/blob/main/cats_and_dog_classification_using_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction: Dog and Cat Classification Using Transfer Learning

## 1. The Problem Statement

Classifying images of dogs and cats is a classic foundational problem in **Computer Vision**. While distinct to the human eye, these two animals share many visual similarities (four legs, fur, tails, similar ear structures).

For a traditional neural network (built from scratch) to learn the difference, it would require:

* A massive dataset (tens of thousands of images).
* Significant computational power (GPUs).
* Hours or days of training time to learn basic features like "what is an edge?" or "what is a circle?" before it even learns "what is a dog?"

## 2. The Solution: Transfer Learning

**Transfer Learning** is a technique where a model developed for a task is reused as the starting point for a model on a second task.

> **The Analogy:** Imagine you want to learn to read medical textbooks in French. You don't start by learning the alphabet (A, B, C) from scratch. You already know how to read; you just "transfer" that reading skill to a new domain (French medical texts).

In Deep Learning, we take a model that has already "learned to read" images (usually on a massive dataset like **ImageNet**, which contains 14 million images and 1,000 classes) and repurpose it to specifically look for dogs and cats.

## 3. How It Works (Under the Hood)

Recall the **Multilayer Perceptron** structure we discussed (Inputs  Hidden Layers  Output). In Transfer Learning, we split the network into two parts:

### Part A: The Pre-Trained "Base" (The Feature Extractor)

We take a famous, powerful architecture (like **VGG16**, **ResNet50**, or **MobileNet**) that has already been trained.

* **The Weights:** These weights are already optimized to detect edges, textures, curves, and complex shapes (like eyes or snouts).
* **The Action:** We **"Freeze"** these layers. We tell our code *not* to update these weights during training. We want to keep this "knowledge" exactly as it is.

### Part B: The Custom "Head" (The Classifier)

We chop off the top layer of the pre-trained model (which was designed to classify 1,000 things) and replace it with our own, simple layers.

* **Input:** The high-level features coming out of the "Base."
* **Hidden Layers:** Maybe one or two small dense layers.
* **Output Layer:** A single node with a **Sigmoid** activation function (since we only have 2 classes: Dog or Cat).

## 4. The Workflow

When we implement this in code, the process follows these standard steps:

1. **Load the Pre-trained Model:** Download a model like `MobileNetV2` without the tomp layer (`include_top=False`).
2. **Freeze Base Layers:** Set `layer.trainable = False` for the loaded model.
3. **Add Custom Layers:** Add a `GlobalAveragePooling` layer, a `Dense` layer (optional), and the final `Output` layer.
4. **Train (Fine-Tune):** Feed in our specific Dog/Cat images. Since the model already knows how to "see," it only needs a few epochs to learn the difference between "Dog features" and "Cat features."

## 5. Summary of Benefits

| Feature | Training from Scratch | Transfer Learning |
| --- | --- | --- |
| **Data Needed** | Massive (10,000+) | Minimal (can work with 100s) |
| **Training Time** | Hours/Days | Minutes |
| **Accuracy** | Often lower (overfitting risk) | Generally very high |
| **Compute Power** | High (needs heavy GPU) | Low (can often run on CPU) |

In [3]:
import os

print(os.listdir(path))

['PetImages']


In [5]:
train_path = os.path.join(path, "PetImages")
print(os.listdir(train_path))

['Dog', 'Cat']


In [7]:
from PIL import Image
import numpy as np

cat_dir = os.path.join(train_path, "Cat")
dog_dir = os.path.join(train_path, "Dog") # Added for consistency, assuming it might be needed later

# Safely get an image path, handling potential empty directories or other issues

img_path = None
if os.path.exists(cat_dir) and os.listdir(cat_dir):
    img_path = os.path.join(cat_dir, os.listdir(cat_dir)[0])
elif os.path.exists(dog_dir) and os.listdir(dog_dir):
    img_path = os.path.join(dog_dir, os.listdir(dog_dir)[0])

if img_path:
    img = Image.open(img_path)
    img = img.resize((224, 224))
    img_array = np.array(img)
    print(f"Successfully loaded and resized image from: {img_path}")
else:
    print(f"No images found in either {cat_dir} or {dog_dir}")

Successfully loaded and resized image from: /kaggle/input/dog-and-cat-classification-dataset/PetImages/Cat/7981.jpg


In [9]:
import tensorflow as tf

# Define image dimensions and batch size
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# Create a data generator for training images with augmentation
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255, # Normalize pixel values to [0,1]
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2 # Use 20% of the data for validation
)

# Create a data generator for validation images (only rescaling)
validation_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Flow training images in batches from the directory
train_generator = train_datagen.flow_from_directory(
    train_path, # Path to the main dataset directory
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training' # Specify this is the training subset
)

# Flow validation images in batches from the directory
validation_generator = validation_datagen.flow_from_directory(
    train_path, # Path to the main dataset directory
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation' # Specify this is the validation subset
)

print(f"Class names: {train_generator.class_indices}")
print(f"Number of training images: {train_generator.samples}")
print(f"Number of validation images: {validation_generator.samples}")

'/kaggle/input/dog-and-cat-classification-dataset/PetImages/Cat'